# Gated Recurrent Unit (GRU)
"Learning Phrase Representations using RNN Encoder-Decoder" (Cho et al., 2014)

**Idea:** a simpler LSTM. Merge the forget and input gates into one **update gate** $z_t$, and drop the separate cell state. The new state is an interpolation between the old state and a candidate.

### GRU vs LSTM vs RNN
| | RNN | LSTM | GRU |
|---|---|---|---|
| State | $h$ | $h$ + $c$ | $h$ only |
| Gates | 0 | 3 (forget, input, output) | 2 (update $z$, reset $r$) |
| Memory update | overwrite | $c_t = f \odot c_{t-1} + i \odot g$ | $h_t = (1 - z) \odot n + z \odot h_{t-1}$ |
| Params per layer | $1\times$ | $4\times$ | $3\times$ |
| Quality | worst | best on long sequences (slightly) | ~ LSTM in practice, cheaper |

Shape flow: `idx (B, T) -> embed (B, T, E) -> loop t: h (B, H) -> stack (B, T, H) -> logits (B, T, V)`

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Data - tiny Shakespeare (character level)
~1.1M characters, vocab = 65 unique characters. Each training example is a random chunk of `block_size` characters; the target is the same chunk shifted by one (predict the next character).

The same data and loss are used in `rnn`, `lstm`, `gru`, `slm`, `moe`, `ssm`, so their val losses can be compared.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text))          # (1115394,)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 64   # T: characters per sequence
batch_size = 64   # B

def get_batch(split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print("vocab size:", vocab_size, "| train chars:", len(train_data), "| val chars:", len(val_data))

## 2. GRU cell
Input and hidden projections are kept separate because the reset gate multiplies only the hidden part of the candidate.

### Formula: Update and reset gates
$$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z), \qquad r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$
- $x_t \in \mathbb{R}^{E}$, $h_{t-1} \in \mathbb{R}^{H}$: input and previous state
- $W_\cdot \in \mathbb{R}^{H \times E}$: input weights (fused in `self.x2h`, $3H$ outputs)
- $U_\cdot \in \mathbb{R}^{H \times H}$: hidden weights (fused in `self.h2h`, $3H$ outputs)
- $z_t$: **update** gate, how much of the old state to keep (plays the role of LSTM's $f$, with $1 - z$ as its $i$)
- $r_t$: **reset** gate, how much of the old state to use when building the candidate

### Formula: Candidate and new state
$$n_t = \tanh\!\left(W_n x_t + b_{in} + r_t \odot (U_n h_{t-1} + b_{hn})\right), \qquad h_t = (1 - z_t) \odot n_t + z_t \odot h_{t-1}$$
- $n_t$: candidate state; $r_t \approx 0$ lets it ignore the past ("start a new phrase")
- $h_t$: convex mix of new and old, per unit. $z_t \approx 1$ copies $h_{t-1}$ unchanged, giving the same gradient highway as the LSTM cell state
- this matches PyTorch's `nn.GRU` convention ($r$ applied after $U_n h$)

In [ ]:
class GRUCell(nn.Module):
    # x_t (B, E), h (B, H) -> h_new (B, H)
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.x2h = nn.Linear(input_size, 3 * hidden_size)   # W_z, W_r, W_n
        self.h2h = nn.Linear(hidden_size, 3 * hidden_size)  # U_z, U_r, U_n

    def forward(self, x, h):
        xz, xr, xn = self.x2h(x).chunk(3, dim=1)  # each (B, H)
        hz, hr, hn = self.h2h(h).chunk(3, dim=1)  # each (B, H)
        z = torch.sigmoid(xz + hz)
        r = torch.sigmoid(xr + hr)
        n = torch.tanh(xn + r * hn)
        return (1 - z) * n + z * h                # (B, H)

## 3. Character-level GRU language model
Exactly the `CharRNN` wrapper with the cell swapped: the state is a single $h$.

In [ ]:
class CharGRU(nn.Module):
    # idx (B, T), h (B, H) or None -> logits (B, T, V), h (B, H)
    def __init__(self, vocab_size, emb_dim=64, hidden_size=256):
        super().__init__()
        self.hidden_size = hidden_size
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.cell = GRUCell(emb_dim, hidden_size)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, h=None):
        B, T = idx.shape
        if h is None:
            h = torch.zeros(B, self.hidden_size, device=idx.device)
        x = self.embed(idx)                # (B, T, E)
        outs = []
        for t in range(T):
            h = self.cell(x[:, t], h)      # (B, H)
            outs.append(h)
        out = torch.stack(outs, dim=1)     # (B, T, H)
        return self.head(out), h           # (B, T, V), (B, H)

model = CharGRU(vocab_size).to(device)
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.3f}M  (~3x the RNN cell)")
logits, h = model(torch.zeros(2, 10, dtype=torch.long, device=device))
print(logits.shape, h.shape)

## 4. Training
Same loop as `rnn.ipynb` and `lstm.ipynb`.

### Formula: Next-token cross-entropy
$$\mathcal{L} = -\frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T} \log p_\theta(x_{t+1} \mid x_{\le t})$$
- $B, T$: batch size and sequence length
- $x_{t+1}$: the true next character
- $p_\theta(\cdot \mid x_{\le t})$: softmax over the vocab from the logits at position $t$
- a model that guesses uniformly scores $\log 65 \approx 4.17$; lower is better

### Formula: Gradient clipping (by global norm)
$$g \leftarrow g \cdot \min\!\left(1, \frac{c}{\lVert g \rVert_2}\right)$$
- $g$: all parameter gradients concatenated into one vector
- $\lVert g \rVert_2$: its L2 norm
- $c$: max norm (1.0); rescales the whole gradient, keeps its direction

In [ ]:
max_iters = 3000
eval_every = 500
lr = 3e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

def loss_fn(model, x, y):
    logits, _ = model(x)                                              # (B, T, V)
    return F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1))  # (B*T, V) vs (B*T,) -> scalar

@torch.no_grad()
def estimate_loss(model, iters=50):
    model.eval()
    out = {s: sum(loss_fn(model, *get_batch(s)).item() for _ in range(iters)) / iters for s in ["train", "val"]}
    model.train()
    return out

for it in range(max_iters + 1):
    if it % eval_every == 0:
        l = estimate_loss(model)
        print(f"iter {it:5d} | train {l['train']:.4f} | val {l['val']:.4f}")
    if it == max_iters:
        break
    x, y = get_batch("train")
    loss = loss_fn(model, x, y)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

## 5. Generation

### Formula: Sampling with temperature
$$p_i = \frac{e^{z_i / \tau}}{\sum_j e^{z_j / \tau}}, \qquad x_{t+1} \sim \text{Categorical}(p)$$
- $z_i$: logit of character $i$
- $\tau$: temperature (0.8); $\tau < 1$ sharpens (safer), $\tau > 1$ flattens (more random), $\tau \to 0$ = greedy argmax

In [ ]:
@torch.no_grad()
def generate(model, start="\n", n=500, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)       # (1, len(start))
    logits, state = model(idx)                                # read the prompt into the state
    out = list(start)
    for _ in range(n):
        probs = F.softmax(logits[:, -1] / temperature, dim=-1)  # (1, V)
        next_id = torch.multinomial(probs, 1)                  # (1, 1)
        out.append(itos[next_id.item()])
        logits, state = model(next_id, state)                  # feed only the new char + carried state
    return "".join(out)

print(generate(model))

## Notes
- Expect val loss close to the LSTM's, with 25% fewer recurrent parameters.
- RNN -> LSTM -> GRU all share the same limit: one step at a time, so training is slow on long sequences and memory is squeezed into a fixed-size $h$.
- The next notebook (**SLM**) drops recurrence entirely: a causal Transformer sees every previous token directly and trains in parallel.
- `nn.GRU(emb_dim, hidden_size, batch_first=True)` is the fused-kernel version of this cell.